# PetFace (ArcFace) 로 보호소 개/고양이 re-ID 테스트

MegaDescriptor 와 **같은 평가 하네스**로 비교한다.

- 모델: PetFace 공식 ArcFace 체크포인트 — torchvision ResNet-50 백본, 512-d 임베딩
- 매칭: `verification.py` 와 동일하게 `F.normalize(backbone(x))` -> 코사인 유사도 (softmax fc 는 seen-id 분류용이라 무시)
- 주의: PetFace 는 **정렬된 얼굴 크롭**으로 학습됨. 우리 데이터는 YOLO 전신 크롭이라 분포가 다르다.
  - **Run A**: 정렬 없이 그대로 넣어보고 신호가 있는지 확인
  - **Run B**: AnyFace 로 5-keypoint 검출 -> 정렬 후 재측정 (Run A 가 가망 있을 때만)
- 참고치(MegaDescriptor, 344개체/1358장): LOO Top-1 **0.717** / split Top-1 **0.528**


In [1]:
# torch / torchvision 는 MegaDescriptor 노트북과 같은 .venv 에 이미 설치돼 있음
%pip install -q gdown

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from PIL import Image
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

ML_DIR     = Path("..")                      # ML/notebooks/ 에서 실행한다고 가정, ML/ 은 한 단계 위
DATA_DIR   = ML_DIR / "dataset" / "raw" / "processed_animals"
WEIGHT_DIR = ML_DIR / "checkpoints" / "petface_pretrained"
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 사전학습 가중치 받기

PetFace 공식 Drive 폴더: <https://drive.google.com/drive/folders/1XZHxlvRUZSQeFrztz0GaKgyVUSCh1lT6>
(**비상업 연구용**)

아래 셀이 폴더 전체를 받으려 시도한다. method x species 조합이 많아 용량이 크니,
`arcface/dog.pt`, `arcface/cat.pt`, (있으면) `unified.pt` 만 수동으로 `ML/checkpoints/petface_pretrained/` 에
넣고 이 셀은 건너뛰어도 된다.

In [ ]:
import gdown

try:
    gdown.download_folder(
        "https://drive.google.com/drive/folders/1XZHxlvRUZSQeFrztz0GaKgyVUSCh1lT6",
        output=str(WEIGHT_DIR),
        quiet=False,
        use_cookies=False,
    )
except Exception as e:
    print("자동 다운로드 실패 - 수동으로 받아서 ML/checkpoints/petface_pretrained/ 에 넣어줘:", e)

print("\n받은 .pt 파일:")
for p in sorted(WEIGHT_DIR.rglob("*.pt")):
    print(" ", p.relative_to(WEIGHT_DIR), f"{p.stat().st_size/1e6:.1f} MB")

In [4]:
# PetFace backbones/resnet.py 의 r50 정의를 그대로 인라인
def r50(n_classes=512):
    model = resnet50(weights=None)          # 학습된 state_dict 로 덮어쓰므로 ImageNet 가중치 불필요
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, n_classes),
        nn.BatchNorm1d(n_classes),
    )
    return model


def load_petface(weight_path):
    backbone = r50()
    ckpt = torch.load(weight_path, map_location="cpu", weights_only=False)
    backbone.load_state_dict(ckpt["state_dict_backbone"])
    return backbone.to(DEVICE).eval()

In [14]:
# 테스트 전처리: verification.py 와 동일 (ToTensor + ImageNet 정규화).
# PetFace 정렬 이미지는 224x224 이고 우리 크롭도 224x224 지만 안전하게 Resize 추가.
petface_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


class FlatImageDataset(Dataset):
    def __init__(self, root, transform):
        self.transform = transform
        self.samples = []
        exts = {".jpg", ".jpeg", ".png", ".webp"}
        for d in sorted(p for p in Path(root).iterdir() if p.is_dir()):
            for f in sorted(d.iterdir()):
                if f.suffix.lower() in exts:
                    self.samples.append((f, d.name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.transform(Image.open(path).convert("RGB")), label


@torch.no_grad()
def extract_petface(root, weight_path, batch_size=64):
    backbone = load_petface(weight_path)
    dl = DataLoader(FlatImageDataset(root, petface_tf), batch_size=batch_size, num_workers=0)

    embs, labels = [], []
    for x, lab in tqdm(dl, desc=f"{Path(weight_path).stem}: {root}"):
        v = F.normalize(backbone(x.to(DEVICE)))
        embs.append(v.cpu().numpy())
        labels.extend(lab)
    return np.concatenate(embs), np.array(labels)

In [15]:
# 평가 하네스 (model_test.ipynb 와 동일)
def evaluate_loo(embeddings, labels):
    labels = np.asarray(labels)
    sim = embeddings @ embeddings.T
    np.fill_diagonal(sim, -np.inf)
    n = len(labels)
    top1 = top5 = 0
    for i in range(n):
        r = np.argsort(sim[i])[::-1]
        top1 += labels[r[0]] == labels[i]
        top5 += np.any(labels[r[:5]] == labels[i])
    return top1 / n, top5 / n


def evaluate_split(embeddings, labels, seed=0):
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    gal = np.array([rng.choice(np.where(labels == l)[0]) for l in np.unique(labels)])
    gs = set(gal.tolist())
    qry = np.array([i for i in range(len(labels)) if i not in gs])
    g_emb, g_lab = embeddings[gal], labels[gal]
    sim = embeddings[qry] @ g_emb.T
    top1 = top5 = 0
    for row, gt in zip(sim, labels[qry]):
        r = np.argsort(row)[::-1]
        top1 += g_lab[r[0]] == gt
        top5 += np.any(g_lab[r[:5]] == gt)
    q = len(qry)
    return top1 / q, top5 / q


def evaluate_split_mean(embeddings, labels, seeds=range(5)):
    t1 = [evaluate_split(embeddings, labels, s)[0] for s in seeds]
    t5 = [evaluate_split(embeddings, labels, s)[1] for s in seeds]
    return (np.mean(t1), np.std(t1)), (np.mean(t5), np.std(t5))

## Run A - 정렬 없이 그대로

`processed_animals` (YOLO 전신 크롭) 를 PetFace ArcFace 에 그대로 통과.
한국 보호소 데이터는 개가 다수라 `dog.pt` 우선, 없으면 `unified.pt`.


In [16]:
cands = ["arcface/dog.pt", "dog.pt", "unified.pt", "arcface/unified.pt", "arcface/cat.pt", "cat.pt"]
WEIGHT = next((WEIGHT_DIR / c for c in cands if (WEIGHT_DIR / c).exists()), None)
assert WEIGHT is not None, f"{WEIGHT_DIR} 안에 dog.pt / unified.pt 를 넣어줘"
print("weight:", WEIGHT)

emb_pf, lab_pf = extract_petface(DATA_DIR, WEIGHT)

t1, t5 = evaluate_loo(emb_pf, lab_pf)
(m1, s1), (m5, s5) = evaluate_split_mean(emb_pf, lab_pf)

print(f"\n[PetFace ArcFace / 정렬X]  개체 {len(set(lab_pf))} / 이미지 {len(lab_pf)}")
print(f"  LOO    Top-1 {t1:.4f} / Top-5 {t5:.4f}")
print(f"  split  Top-1 {m1:.4f}+-{s1:.4f} / Top-5 {m5:.4f}+-{s5:.4f}")
print(f"\n[참고 MegaDescriptor]  LOO Top-1 0.717 / split Top-1 ~0.528")

weight: petface_pretrained\dog.pt


dog: processed_animals: 100%|██████████| 22/22 [00:02<00:00,  7.67it/s]



[PetFace ArcFace / 정렬X]  개체 344 / 이미지 1358
  LOO    Top-1 0.8041 / Top-5 0.9205
  split  Top-1 0.6627+-0.0059 / Top-5 0.8333+-0.0068

[참고 MegaDescriptor]  LOO Top-1 0.717 / split Top-1 ~0.528


In [ ]:
emb_pfm, lab_pfm = extract_petface(ML_DIR / "dataset" / "raw" / "processed_animals_masked", WEIGHT)
t1, t5 = evaluate_loo(emb_pfm, lab_pfm)
(m1, s1), (m5, s5) = evaluate_split_mean(emb_pfm, lab_pfm)
print(f"[PetFace / 배경제거]  개체 {len(set(lab_pfm))} / 이미지 {len(lab_pfm)}")
print(f"  LOO    Top-1 {t1:.4f} / Top-5 {t5:.4f}")
print(f"  split  Top-1 {m1:.4f}+-{s1:.4f} / Top-5 {m5:.4f}+-{s5:.4f}")

### 해석 가이드

- split Top-1 이 **0.1 미만** -> 전신 크롭은 완전히 out-of-distribution. 정렬(Run B) 필수.
- **0.1 ~ 0.3** -> 얼굴 신호가 조금 잡힘. 정렬하면 크게 오를 여지.
- **0.4+** -> 정렬 없이도 쓸만. 정렬로 더 밀어붙일 가치 큼.

PetFace 는 얼굴(배경 없는 고신호 영역) 기반이라, 얼굴만 제대로 잡히면
MegaDescriptor 가 겪은 "배경이 점수 부풀림" 문제를 구조적으로 피한다.


## Run B - AnyFace 정렬 후 재측정  *(Run A 가 가망 있을 때만)*

AnyFace: <https://github.com/IS2AI/AnyFace> - 동물 얼굴 5-keypoint 검출기 (별도 모델/가중치).
`detect_5kpts` 를 AnyFace 추론 코드로 채운 뒤 실행.


In [ ]:
# PetFace repo 에서 정렬용 소스 keypoint 만 가져온다
#   git clone --depth 1 https://github.com/mapooon/PetFace ../external/petface_repo
import cv2
import skimage.transform

PETFACE_REPO = ML_DIR / "external" / "petface_repo"
SRC_KPTS = {
    "dog": np.load(PETFACE_REPO / "keypoints" / "dog.npy").reshape(5, 2),
    "cat": np.load(PETFACE_REPO / "keypoints" / "cat.npy").reshape(5, 2),
}


def align_face(bgr_img, lmk_5x2, species="dog", size=224):
    tf = skimage.transform.SimilarityTransform()
    tf.estimate(np.asarray(lmk_5x2, dtype=np.float64), SRC_KPTS[species])
    return cv2.warpPerspective(bgr_img, tf.params.copy(), (size, size), flags=cv2.INTER_AREA)


def detect_5kpts(bgr_img):
    # AnyFace 로 5점 검출 -> np.ndarray (5,2). 실패 시 None.
    # AnyFace repo 의 추론 코드로 채울 것.
    raise NotImplementedError("AnyFace 추론 코드 연결 필요")


FACE_DIR = ML_DIR / "dataset" / "raw" / "processed_animals_faces"
FACE_DIR.mkdir(parents=True, exist_ok=True)

n_ok = n_fail = 0
for animal_dir in tqdm(sorted(p for p in DATA_DIR.iterdir() if p.is_dir())):
    out_dir = FACE_DIR / animal_dir.name
    out_dir.mkdir(exist_ok=True)
    for f in sorted(animal_dir.glob("*.jpg")):
        img = cv2.imread(str(f))
        kpts = detect_5kpts(img)
        if kpts is None:
            n_fail += 1
            continue
        cv2.imwrite(str(out_dir / f.name), align_face(img, kpts, species="dog"))
        n_ok += 1

# 2장 이하 남은 개체 폴더 정리
for d in list(FACE_DIR.iterdir()):
    if d.is_dir() and len(list(d.glob("*.jpg"))) <= 2:
        for f in d.glob("*.jpg"):
            f.unlink()
        d.rmdir()

print(f"정렬 성공 {n_ok} / 실패 {n_fail} / 남은 개체 {sum(1 for p in FACE_DIR.iterdir() if p.is_dir())}")

In [18]:
# 정렬된 얼굴로 재측정
emb_face, lab_face = extract_petface(FACE_DIR, WEIGHT)

t1, t5 = evaluate_loo(emb_face, lab_face)
(m1, s1), (m5, s5) = evaluate_split_mean(emb_face, lab_face)
print(f"[PetFace ArcFace / 정렬O]  개체 {len(set(lab_face))} / 이미지 {len(lab_face)}")
print(f"  LOO    Top-1 {t1:.4f} / Top-5 {t5:.4f}")
print(f"  split  Top-1 {m1:.4f}+-{s1:.4f} / Top-5 {m5:.4f}+-{s5:.4f}")

dog: processed_animals_faces: 0it [00:00, ?it/s]


ValueError: need at least one array to concatenate